# PR17 pretrained OCR benchmark

This owner-operated notebook benchmarks Tesseract 5, EasyOCR and PaddleOCR on the private controlled-real validation partition. It does not train an OCR recognizer and cannot load the locked Ghana test. Reports stay in private Drive storage.


In [ ]:
RUN_PROFILE = "smoke"  # validation benchmark only; no recognizer training
TARGET_COMMIT = "ed961748fec12503550b387fc0c7e9187781cafa"
REPOSITORY_URL = "https://github.com/davidagyekum/momo-fraud-detection.git"
DRIVE_ROOT = "/content/drive/MyDrive/momo-fraud"
VM_ROOT = "/content/momo-work"
NOTEBOOK_PATH = "ml/notebooks/colab/06_benchmark_ocr.ipynb"
EXPECTED_PRIVATE_SPLIT_SHA256 = "3c2bd2e3727b62f0a61f01a7eebcbe49da7ed0ac124a8f765d471533d867d941"
EXPECTED_DEVELOPMENT_MANIFEST_SHA256 = "1ba8c58e1c29b77a46ba3cc54da7843dd84f090fdb2e731704091162d556d644"
EXPECTED_PRIVATE_ARCHIVE_SHA256 = "8a7f4b58e20569775dc237e5e4fefba78e2bc5aa8c40d5ee41dd893d495ea9b9"
RUN_BENCHMARK = False  # change to True only after the private development bundle is uploaded
assert RUN_PROFILE == "smoke"
assert len(TARGET_COMMIT) == 40 and TARGET_COMMIT != "REPLACE_WITH_PUSHED_PR17_SHA"


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import runpy
import subprocess
import sys
import zipfile
from google.colab import drive

drive.mount("/content/drive", force_remount=True, timeout_ms=600000)
repo = Path(VM_ROOT) / "repo"
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "fetch", "--prune", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", TARGET_COMMIT], check=True)
bootstrap_path = repo / "ml/src/momo_fdvs_ml/colab_bootstrap.py"
bootstrap = runpy.run_path(str(bootstrap_path))
versions_before_install = bootstrap["distribution_snapshot"]()
modules_before_install = tuple(sys.modules)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(repo / "ml/requirements-ocr.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(repo / "ml")], check=True)
versions_after_install = bootstrap["distribution_snapshot"]()
current_process_probe = bootstrap["runtime_import_probe"]()
child_process_probe = subprocess.run([sys.executable, str(bootstrap_path), "--probe"], check=False, capture_output=True, text=True)
child_process_report = json.loads(child_process_probe.stdout)
if child_process_probe.returncode != 0 or child_process_report["healthy"] is not True:
    print(json.dumps({"runtime_import_probe": child_process_report, "locked_test_accessed": False, "training_executed": False}, indent=2, sort_keys=True))
    raise RuntimeError("PINNED_RUNTIME_IMPORT_PROBE_FAILED: use Runtime > Disconnect and delete runtime, then run all cells again")
restart_report = bootstrap["runtime_restart_report"](before=versions_before_install, after=versions_after_install, loaded_modules=modules_before_install, current_process_healthy=current_process_probe["healthy"] is True)
if restart_report["restart_required"] is True:
    print(json.dumps({"restart_report": restart_report, "locked_test_accessed": False, "training_executed": False}, indent=2, sort_keys=True))
    raise RuntimeError("COLAB_RUNTIME_RESTART_REQUIRED: use Runtime > Restart session, then run all cells again")
ocr_bootstrap_path = repo / "ml/src/momo_fdvs_ml/colab_ocr.py"
ocr_bootstrap = runpy.run_path(str(ocr_bootstrap_path))
tesseract_report = ocr_bootstrap["ensure_tesseract5"](source_root=Path(VM_ROOT) / "cache/tesseract-5.5.3")
assert tesseract_report["required_major_version"] == 5
sys.path.insert(0, str(repo / "ml/src"))
os.environ["PADDLE_PDX_CACHE_HOME"] = str(Path(DRIVE_ROOT) / "cache/ocr/paddlex")


In [ ]:
from momo_fdvs_ml.colab import ColabPaths, colab_preflight_report
from momo_fdvs_ml.execution import ExecutionProfile
from momo_fdvs_ml.ocr_benchmark import engine_inventory, load_ocr_benchmark_config, load_ocr_development_bundle

paths = ColabPaths(drive_root=Path(DRIVE_ROOT), vm_root=Path(VM_ROOT))
preflight = colab_preflight_report(repo, paths=paths, profile=ExecutionProfile.SMOKE, notebook=NOTEBOOK_PATH, require_colab=True)
assert preflight["git"]["commit"] == TARGET_COMMIT
assert preflight["git"]["dirty"] is False
private_archive = Path(DRIVE_ROOT) / "private-governance/ghana-private/pr17-ocr-development.zip"
archive_sha256 = hashlib.sha256(private_archive.read_bytes()).hexdigest()
assert archive_sha256 == EXPECTED_PRIVATE_ARCHIVE_SHA256
private_bundle = Path(VM_ROOT) / "private/pr17-ocr-development"
private_bundle.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(private_archive) as archive:
    raw_members = archive.namelist()
    assert all("\\" not in name for name in raw_members), "archive members must use POSIX separators"
    members = [PurePosixPath(name) for name in raw_members]
    assert all(not member.is_absolute() and ".." not in member.parts for member in members)
    archive.extractall(private_bundle)
development_manifest = private_bundle / "development-manifest.json"
config = load_ocr_benchmark_config(repo / "ml/configs/ocr_benchmark_v2.json")
validation_records = load_ocr_development_bundle(development_manifest, partition="validation")
bundle_manifest = json.loads(development_manifest.read_text(encoding="utf-8"))
assert bundle_manifest["manifest_sha256"] == EXPECTED_DEVELOPMENT_MANIFEST_SHA256
assert bundle_manifest["source_split_manifest_sha256"] == EXPECTED_PRIVATE_SPLIT_SHA256
assert bundle_manifest["locked_test_included"] is False
print(json.dumps({"commit": TARGET_COMMIT, "validation_records": len(validation_records), "engine_inventory": engine_inventory(), "tesseract_bootstrap": tesseract_report, "locked_test_accessed": False, "training_executed": False}, indent=2, sort_keys=True))


In [ ]:
from momo_fdvs_ml.ocr_benchmark import run_ocr_parser_ceiling_diagnostic

run_root = Path(DRIVE_ROOT) / f"runs/pr17-ocr-{TARGET_COMMIT[:8]}"
run_root.mkdir(parents=True, exist_ok=True)
parser_ceiling_path = run_ocr_parser_ceiling_diagnostic(development_manifest_path=development_manifest, output_path=run_root / "ocr-parser-ceiling-report.json", repository_root=repo)
parser_ceiling = json.loads(parser_ceiling_path.read_text(encoding="utf-8"))
assert parser_ceiling["raw_text_persisted"] is False
assert parser_ceiling["field_values_persisted"] is False
assert parser_ceiling["record_identifiers_persisted"] is False
assert parser_ceiling["locked_test_accessed"] is False
print(json.dumps(parser_ceiling, indent=2, sort_keys=True))


In [ ]:
from momo_fdvs_ml.ocr_benchmark import EasyOCRAdapter, OCRBenchmarkError, OCRConfiguration, PaddleOCRAdapter, TesseractAdapter

assert RUN_BENCHMARK is True, "Set RUN_BENCHMARK=True in the first code cell after reviewing the private bundle summary."
adapters = {}
adapter_failures = {}
try:
    adapters["tesseract"] = TesseractAdapter(psm=6, language="eng")
except OCRBenchmarkError:
    adapter_failures["tesseract"] = "OCR_ENGINE_UNAVAILABLE"
try:
    adapters["easyocr"] = EasyOCRAdapter(gpu=False, model_storage_directory=str(Path(DRIVE_ROOT) / "cache/ocr/easyocr"))
except OCRBenchmarkError:
    adapter_failures["easyocr"] = "OCR_ENGINE_UNAVAILABLE"
try:
    adapters["paddleocr"] = PaddleOCRAdapter(device="cpu", ocr_version="PP-OCRv6", enable_mkldnn=False)
except OCRBenchmarkError:
    adapter_failures["paddleocr"] = "OCR_ENGINE_UNAVAILABLE"
configurations = tuple(OCRConfiguration(engine["engine"], variant, engine["options"]) for engine in config["engines"] for variant in config["preprocessing_variants"])
print(json.dumps({"adapter_failures": adapter_failures, "configuration_count": len(configurations)}, indent=2, sort_keys=True))


In [ ]:
from momo_fdvs_ml.ocr_benchmark import run_ocr_validation_benchmark, select_engine_finalists

screen_report = run_ocr_validation_benchmark(development_manifest_path=development_manifest, configurations=configurations, adapters=adapters, output_path=run_root / "ocr-screen-report.json", repository_root=repo, source_group_limit=config["data_policy"]["screen_source_group_limit"])
finalists = select_engine_finalists(screen_report)
print(json.dumps({"screen_report": str(screen_report), "finalists": [item.configuration_id for item in finalists], "locked_test_accessed": False}, indent=2, sort_keys=True))


In [ ]:
import hashlib
from momo_fdvs_ml.ocr_benchmark import load_selected_ocr_bundle, select_ocr_configuration

full_report = run_ocr_validation_benchmark(development_manifest_path=development_manifest, configurations=finalists, adapters=adapters, output_path=run_root / "ocr-validation-report.json", repository_root=repo)
selected_path = select_ocr_configuration(report_path=full_report, output_path=run_root / "selected-ocr-bundle.json", repository_root=repo)
selected = load_selected_ocr_bundle(selected_path)
safe_summary = {"report_sha256": json.loads(full_report.read_text(encoding="utf-8"))["report_sha256"], "bundle_sha256": selected["bundle_sha256"], "status": selected["status"], "engine": selected["engine"], "variant": selected["variant"], "release_gates": selected["validation_metrics"]["release_gates"], "promotable": selected["promotable"], "locked_test_accessed": selected["locked_test_accessed"], "training_executed": selected["training_executed"]}
assert safe_summary["locked_test_accessed"] is False
assert safe_summary["training_executed"] is False
print(json.dumps(safe_summary, indent=2, sort_keys=True))


## Stop boundary

Stop after the selected validation bundle and safe summary are written. Do not open the five-record Ghana test, do not tune after test access, do not train an OCR recognizer, and do not promote a configuration whose validation gates failed. The missing tampered-derivative slice remains an explicit blocker.
